# Validation tuning — seed 0

GPU0: GL lambda0.1,1,10. GPU1: MambaCL+GL lambda0.1,1,10. Each candidate uses the first three tasks, 50 epochs, stratified10% training holdout. Max validation AIA, ties use smaller lambda. Repeat with previous Output until COMPLETE.

Same setup flow as H4: Internet ON, Kaggle Secret `github_token`, T4 x2. Source is cloned from GitHub and checked out at a fixed commit. Attach raw ImageNet-R (for example my1nonly/imagenet-r) and official pretrained weights. The notebook creates train/val symlinks using the pinned Mamba-CL lists, checks all24000/6000 files and200 classes, and also accepts an existing official split. Previous Output is for resume and for carrying01 selection into02. No source ZIP is required.

Active CPU checks pass and are recomputed. CUDA compatibility, VRAM and throughput remain unmeasured.

In [ ]:
BRANCH = 'GL'
REPO_COMMIT = '54c859d0684f7b96bc3460e5c48176ac3c9f6e99'
RUN_TAG = 'glcl_imagenetr_s0_54c859d0684f'
STAGE = 'tune'
DATA_ROOT = None  # optional raw OR official train/val root; auto-discovered under /kaggle/input
PRETRAINED = None  # optional explicit path to defocus_mamba_large_cls_21k.pth
JOBS = [[{'name': 'GL_lambda0.1', 'arm': 'GL', 'strength': 0.1, 'stage': 'tune'}, {'name': 'GL_lambda1', 'arm': 'GL', 'strength': 1.0, 'stage': 'tune'}, {'name': 'GL_lambda10', 'arm': 'GL', 'strength': 10.0, 'stage': 'tune'}], [{'name': 'MambaCL+GL_lambda0.1', 'arm': 'MambaCL+GL', 'strength': 0.1, 'stage': 'tune'}, {'name': 'MambaCL+GL_lambda1', 'arm': 'MambaCL+GL', 'strength': 1.0, 'stage': 'tune'}, {'name': 'MambaCL+GL_lambda10', 'arm': 'MambaCL+GL', 'strength': 10.0, 'stage': 'tune'}]]
import time
SESSION_START = time.time()

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

os.environ.update(TQDM_DISABLE='1', PYTHONUNBUFFERED='1',
                  PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python')
MAIN_URL = 'https://github.com/duyh80456-code/Gramian-Laplace-GL-cho-continual-learning-tr-n-Mamba.git'
WORKING = Path('/kaggle/working')
# A previous execution leaves the kernel inside REPO. Always leave that tree
# before removing it, and also recover if a prior run already deleted the cwd.
WORKING.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING)
REPO = Path('/kaggle/working/gramian-laplace')
RUN_ROOT = Path('/kaggle/working/glcl')
OUTPUT = RUN_ROOT / 'glcl_outputs'
ROOT_OUT = OUTPUT

def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)

def private_clone(url, destination, commit):
    token = UserSecretsClient().get_secret('github_token').strip()
    if not token:
        raise RuntimeError('Enable Kaggle Secret github_token for this notebook')
    askpass = Path('/kaggle/working/.glcl_git_askpass.py')
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = dict(os.environ, GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
    try:
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(destination)], cwd=destination.parent, env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'fetch', 'origin', commit], cwd=destination.parent, env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], cwd=destination.parent, env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

# Only refresh the source checkout. Outputs and environment live elsewhere.
if REPO.exists():
    shutil.rmtree(REPO)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], cwd=WORKING, text=True).strip()
if CODE_COMMIT != REPO_COMMIT:
    raise RuntimeError('Pinned repository revision mismatch')
os.chdir(REPO)
print('Resolved code commit:', CODE_COMMIT, 'pin:', REPO_COMMIT)
print('Official MambaCL:', json.loads((REPO/'upstream/LOCK.json').read_text()))

In [ ]:
# Isolated Python 3.11 + official torch 2.2.2; no Kaggle host torch replacement.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.5.31'], check=True)
VENV = Path('/kaggle/working/glcl_env')
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                '-r', str(REPO/'requirements-kaggle.txt'), 'pip==24.2'], check=True)

ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1', TQDM_DISABLE='1')

In [ ]:
# Recompute the supervisor-corrected section-six checks, same fixture/noises:
# C .852433 > proxy .725509 and >=.8; Delta .968065 > proxy .950543 and >=.85.
# Test5 remains DEFERRED until phase two.
# A failure intentionally stops before model initialization/training.
subprocess.run([PYTHON, '-m', 'tests.section6'], env=ENV, check=True)
print((REPO/'reports/section6.json').read_text())

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
subprocess.run([PYTHON, '-c', "from glcl.kaggle import restore; restore('/kaggle/input', '/kaggle/working/glcl/glcl_outputs')"], env=ENV, check=True)
commit_path = OUTPUT/'code_commit.json'
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append(dict(timestamp=datetime.now(timezone.utc).isoformat(), pin=REPO_COMMIT, commit=CODE_COMMIT, stage=STAGE))
atomic_json(sessions, commit_path)
DATA_REPORT = OUTPUT/'data_preparation.json'
prepare = [PYTHON, 'scripts/prepare_imagenet_r.py', '--input', str(DATA_ROOT or '/kaggle/input'),
           '--stage', str(RUN_ROOT/'input_data'/'imagenet_r'), '--report', str(DATA_REPORT)]
subprocess.run(prepare, env=ENV, check=True)
prepared = json.loads(DATA_REPORT.read_text())
if prepared['status'] != 'PASSED':
    raise RuntimeError('ImageNet-R preparation failed; training blocked')
DATA_ROOT = prepared['data_dir']
if PRETRAINED is None:
    PRETRAINED = subprocess.check_output(
        [PYTHON, 'scripts/prepare_imagenet_r.py', '--find-checkpoint', '/kaggle/input'],
        env=ENV, text=True).strip()
print('Data:', DATA_ROOT, 'Weights:', PRETRAINED)

In [ ]:
# Build for T4 (sm75) using Kaggle's CUDA toolkit. Compilation has not been
# validated in this local CPU environment; failures must stop this notebook.
env = dict(os.environ, TORCH_CUDA_ARCH_LIST='7.5', MAX_JOBS='2',
           MAMBA_FORCE_BUILD='TRUE', CAUSAL_CONV1D_FORCE_BUILD='TRUE')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-build-isolation',
                'causal-conv1d==1.4.0', 'mamba-ssm==2.2.2'], env=env, check=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1',
           PYTHONUNBUFFERED='1', TQDM_DISABLE='1')
subprocess.run([PYTHON, '-c', "import torch; print(torch.__version__); print([(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]); assert torch.cuda.device_count()==2"], env=ENV, check=True)
# Syntax/import only; no image forward or end-to-end smoke test.
subprocess.run([PYTHON, '-c', "import sys; sys.path.insert(0,'upstream/mamba-cl'); import train_eval; from utils.mamba_block import is_fast_path_available; assert is_fast_path_available"], env=ENV, check=True)

In [ ]:
DEADLINE = SESSION_START + 11*3600  # includes environment setup; 5-min hard grace
jobs_path = RUN_ROOT/'effective_jobs'/f'{STAGE}.json'
atomic_json(JOBS, jobs_path)
atomic_json(dict(run_tag=RUN_TAG, stage=STAGE, seed=0, source_commit=CODE_COMMIT,
                 jobs=JOBS, workers=2, OMP_NUM_THREADS=1, MKL_NUM_THREADS=1,
                 protocol='Full registered horizon; exit75 resumes without truncation'),
            OUTPUT/f'notebook_protocol_{STAGE}.json')
command = [PYTHON, '-m', 'glcl.kaggle', '--stage', STAGE, '--root', str(OUTPUT),
           '--data', DATA_ROOT, '--pretrained', PRETRAINED, '--deadline', str(DEADLINE), '--jobs', str(jobs_path)]
result = subprocess.run(command, env=ENV)
if result.returncode not in (0, 75):
    raise RuntimeError('Run failed. Inspect glcl_outputs/*/*/console.log')
print('PAUSED: attach this Output and rerun same notebook' if result.returncode == 75 else 'Stage complete')

In [ ]:
for path in (OUTPUT/'selection.json', OUTPUT/'session.json', OUTPUT/'comparison.csv', OUTPUT/'comparison.md'):
    if path.exists():
        print(path.name, path.read_text())
print('Archive: /kaggle/working/glcl/glcl_outputs.tar.gz; attach it as previous Output to resume.')